# Content Moderation at Scale

Companion notebook for the [Content Moderation at Scale lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/21-content-moderation).

We implement focal loss for class imbalance, Cohen's kappa for annotator agreement, and an uncertainty-sampling active learning loop. Pure NumPy.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

## 1 — Focal loss for extreme class imbalance

Content moderation has extreme imbalance (< 1% violation rate). Focal loss down-weights easy negatives so the model focuses on hard examples.

In [ ]:
def focal_loss(p, y, gamma=2.0, eps=1e-7):
    """
    p: predicted probabilities (N,)
    y: true labels 0/1 (N,)
    gamma: focusing parameter (0 = standard BCE, 2 = typical focal)
    """
    p = np.clip(p, eps, 1-eps)
    p_t = np.where(y == 1, p, 1-p)
    loss = -((1 - p_t)**gamma) * np.log(p_t)
    return loss.mean()

# Demonstration: easy negative vs hard negative
p_easy_neg = 0.02   # model correctly predicts ~0 for easy safe content
p_hard_neg = 0.45   # model is uncertain about borderline content

y_neg = 0  # true label: not violating
bce_easy  = -np.log(1 - p_easy_neg)
bce_hard  = -np.log(1 - p_hard_neg)
fl_easy   = ((1 - (1 - p_easy_neg))**2) * (-np.log(1 - p_easy_neg))
fl_hard   = ((1 - (1 - p_hard_neg))**2) * (-np.log(1 - p_hard_neg))

print(f"{'':20s} {'BCE loss':>12s}  {'Focal loss (γ=2)':>16s}")
print(f"Easy negative (p=0.02): {bce_easy:12.4f}  {fl_easy:16.4f}")
print(f"Hard negative (p=0.45): {bce_hard:12.4f}  {fl_hard:16.4f}")
print(f"Ratio hard/easy:        {bce_hard/bce_easy:12.1f}x  {fl_hard/fl_easy:14.1f}x")
print("Focal loss up-weights hard examples relative to easy ones")

## 2 — Cohen's kappa for inter-annotator agreement

In [ ]:
def cohens_kappa(labels_a, labels_b):
    """
    labels_a, labels_b: arrays of 0/1 labels from two annotators
    Returns Cohen's kappa coefficient.
    """
    assert len(labels_a) == len(labels_b)
    n = len(labels_a)
    # Observed agreement
    p_o = (labels_a == labels_b).mean()
    # Expected agreement (by chance)
    p_a1 = labels_a.mean()    # annotator A's rate of labeling 1
    p_b1 = labels_b.mean()    # annotator B's rate of labeling 1
    p_e = p_a1*p_b1 + (1-p_a1)*(1-p_b1)
    return (p_o - p_e) / (1 - p_e + 1e-9)

# Simulate two annotators on 200 content items
n_items = 200
true_labels = (rng.random(n_items) < 0.1).astype(int)   # 10% violating

# Annotator A: good but noisy
a_labels = true_labels.copy()
flip_a = rng.random(n_items) < 0.1                       # 10% noise
a_labels[flip_a] = 1 - a_labels[flip_a]

# Annotator B: noisier
b_labels = true_labels.copy()
flip_b = rng.random(n_items) < 0.25                      # 25% noise
b_labels[flip_b] = 1 - b_labels[flip_b]

kappa = cohens_kappa(a_labels, b_labels)
print(f"Cohen's kappa: {kappa:.3f}")
print(f"Interpretation: {'Moderate' if 0.4 < kappa < 0.6 else 'Poor' if kappa < 0.4 else 'Substantial'} agreement")
print(f"Raw agreement: {(a_labels == b_labels).mean():.3f}")

## 3 — Active learning: uncertainty sampling

In [ ]:
# Simulate a pool of 1000 unlabeled items with model confidence scores
n_pool = 1000
# Model outputs probabilities (simulated)
probs = rng.beta(0.5, 0.5, n_pool)  # bimodal: most items confidently classified
# Uncertainty = entropy = -p*log(p) - (1-p)*log(1-p)
eps = 1e-7
entropy = -(probs + eps)*np.log(probs + eps) - (1-probs+eps)*np.log(1-probs+eps)

# Uncertainty sampling: label the 50 most uncertain items first
k = 50
uncertain_ids = np.argsort(-entropy)[:k]
random_ids    = rng.choice(n_pool, k, replace=False)

print(f"Uncertainty-sampled items: mean entropy = {entropy[uncertain_ids].mean():.3f}")
print(f"Random-sampled items:      mean entropy = {entropy[random_ids].mean():.3f}")
print(f"Ratio: {entropy[uncertain_ids].mean()/entropy[random_ids].mean():.1f}x more informative")

## ✏️ Your turn

**Exercise.** Implement `f_beta(precision, recall, beta)` where $\beta > 1$ weights recall more than precision (appropriate when missing violations is more costly than a false alarm).

In [ ]:
def f_beta(precision, recall, beta=2.0):
    """F-beta score. beta > 1 weights recall more than precision."""
    # TODO(you): implement F_beta = (1 + beta^2) * P * R / (beta^2 * P + R)
    return ...

# Compare F1 (beta=1) vs F2 (beta=2) for a high-recall, low-precision model
p, r = 0.20, 0.90          # catches most violations but many false alarms
print(f"F1  (equal weight): {f_beta(p, r, beta=1.0):.4f}")
print(f"F2  (recall 4x):    {f_beta(p, r, beta=2.0):.4f}")
print(f"F0.5 (precision 4x):{f_beta(p, r, beta=0.5):.4f}")

In [ ]:
# Assertion
assert abs(f_beta(0.5, 0.5, 1.0) - 0.5) < 1e-6, "F1 of equal P=R=0.5 should be 0.5"
assert f_beta(0.2, 0.9, 2.0) > f_beta(0.2, 0.9, 1.0), "F2 should score higher-recall model higher than F1"
print(f"✓ f_beta correct: F2={f_beta(0.2, 0.9, 2.0):.4f}")

<details><summary>Solution</summary>

```python
def f_beta(precision, recall, beta=2.0):
    b2 = beta ** 2
    return (1 + b2) * precision * recall / (b2 * precision + recall + 1e-9)
```
</details>